# Your own Colab copy

Choose File, Save a copy in Drive before working. Run the setup cell once, then work through the starter. Download your notebook and exported ZIP before disconnecting. Colab files can disappear when a runtime ends. Submit through the hub after finishing your attempt.

In [ ]:
from pathlib import Path
import hashlib
import os
import urllib.request
import zipfile

work = Path('/content/icaic-unet') if Path('/content').exists() else Path.cwd() / 'icaic-unet'
work.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')
payload = urllib.request.urlopen('https://elteomc.github.io/icaic-practice-hub/packs/unet.zip', timeout=90).read()
assert hashlib.sha256(payload).hexdigest() == 'a2e4bd90df55ebb1e2188fd94279fb1bc4f50e87c41e513d20661a11f2ecd7d0', 'The task pack changed. Reopen the latest notebook from the hub.'
archive_path = work / 'task.zip'
archive_path.write_bytes(payload)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(work)
os.chdir(work / 'unet')
import sys
sys.path.insert(0, str(Path.cwd()))
print('Task folder:', Path.cwd())


# Debugging U-Net

Open submission.py in a second JupyterLab tab. Edit that file, save it, and rerun these checks. The provided functions intentionally need your repair. Read Statement.md for the source correction and exact requirements.

In [ ]:
import importlib
import torch
import submission
torch.set_num_threads(2)
importlib.reload(submission)
model = submission.BrokenUNet()
try:
    with torch.no_grad():
        output = model(torch.randn(1, 3, 256, 256))
    print('Output shape:', tuple(output.shape))
    assert tuple(output.shape) == (1, 2, 110, 110)
except Exception as error:
    print(type(error).__name__ + ':', error)

## Check center cropping

The values matter as well as the shape. A resize is not a crop.

In [ ]:
importlib.reload(submission)
x = torch.arange(63).reshape(1, 1, 7, 9)
try:
    cropped = submission.BrokenUNet().center_crop(x, (1, 1, 3, 5))
    assert torch.equal(cropped, x[:, :, 2:5, 2:7])
    print('Center crop passed')
except Exception as error:
    print(type(error).__name__ + ':', error)

## Check the metric

The second case below corrects an error in the original source self-check. A predicted class with no true pixels still has a nonzero union.

In [ ]:
importlib.reload(submission)
cases = [(torch.tensor([[[0, 0], [1, 1]]]), torch.tensor([[[0, 1], [1, 1]]]), 7 / 12),
         (torch.tensor([[[0, 0], [0, 1]]]), torch.zeros((1, 2, 2), dtype=torch.long), 0.375),
         (torch.zeros((1, 2, 2), dtype=torch.long), torch.zeros((1, 2, 2), dtype=torch.long), 1.0)]
for predicted, truth, expected in cases:
    try:
        actual = float(submission.calculate_miou(predicted, truth, 2))
        print('Got', actual, 'expected', expected)
        assert abs(actual - expected) < 1e-6
    except Exception as error:
        print(type(error).__name__ + ':', error)

## Submit and explain

Save submission.py, then upload it on the hub task page. Keep your reasoning here: why was the skip larger, what does cropping discard, and why is accuracy insufficient for this task?

## Download for the hub

Run this export cell to download your ZIP. Open Submit & notes on the task page and follow the available grading instructions. If you are practicing independently, finish your attempt before checking the final practice score.

In [ ]:
from pathlib import Path
import zipfile
assert Path('submission.py').is_file(), 'Save submission.py first'
with zipfile.ZipFile('hub-upload.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.py', 'submission.py')
output = 'hub-upload.zip'
try:
    from google.colab import files
    files.download(output)
except ImportError:
    print('Upload this file:', Path(output).resolve())
